# v4 Phase 1a — Plain MAPPO check: evaluation

Plan: `PLAN.md`. Training: `train_runner.py` (laptop, Stage 2 = 5 drones, 5 obstacles, seeds 42–46).

| Run | What |
|---|---|
| Fixed-weight MAPPO (laptop) | unchanged v3 code, α = 0.5 — same-hardware reference |
| Plain MAPPO | combined reward 0.5·r_mission + 0.5·r_safety; the actor uses the plain combined-reward advantage (no mission/safety mixing) |
| Plain MAPPO, α = 0.9 reward | as Plain MAPPO, combined reward 0.9·r_mission + 0.1·r_safety |

Evaluation protocol = Phase 0 / `v3/results-table`: deterministic policy, 200 episodes, env seed 7001,
through `../common/v3_env.py`. **Gate 3:** the laptop Fixed-weight MAPPO must land in the range of the
original Kaggle Fixed-weight MAPPO (89.2 ± 2.0% over 5 seeds) before anything else is read.

In [1]:
# Cell 1 -- setup
import os, sys, json, math
import numpy as np, pandas as pd, torch
from scipy import stats
torch.set_num_threads(1)
sys.path.insert(0, os.path.abspath('../common')); sys.path.insert(0, os.path.abspath('../00-diagnostics'))
import v3_registry as R, diag as D

OUT = 'output'; os.makedirs(OUT, exist_ok=True)
N_EVAL, EVAL_SEED = 200, 7001
SEEDS = (42, 43, 44, 45, 46)
RUNS = {'fixed-weight-mappo-laptop': 'Fixed-weight MAPPO (laptop)',
        'plain-mappo': 'Plain MAPPO',
        'plain-mappo-alpha09': 'Plain MAPPO, α = 0.9 reward'}
S2 = dict(n_drones=5, n_obstacles=5, use_hungarian=True)

def entry(run, seed):
    d = os.path.join('runs', run, 'output-seed{}'.format(seed))
    return dict(stage=2, method='B4', seed=seed, run=run, run_dir=d, **S2)   # 'B4' key only selects the fixed-α code path in diag (α is not used in evaluation)

missing = [os.path.join(entry(r, s)['run_dir'], 'final_model.pt') for r in RUNS for s in SEEDS
           if not os.path.exists(os.path.join(entry(r, s)['run_dir'], 'final_model.pt'))]
assert not missing, 'missing models:\n' + '\n'.join(missing)
print('all', len(RUNS) * len(SEEDS), 'trained models found')

all 15 trained models found


In [2]:
# Cell 2 -- evaluate every model (cached)
EVAL_CSV = OUT + '/eval_episodes.csv'
if os.path.exists(EVAL_CSV):
    ev = pd.read_csv(EVAL_CSV, keep_default_na=False); print('loaded cached', EVAL_CSV)
else:
    rows = []
    for run in RUNS:
        for s in SEEDS:
            e = entry(run, s)
            actor, pah = D.load_checkpoint(os.path.join(e['run_dir'], 'final_model.pt'))
            env = D.make_env(e, EVAL_SEED)
            for k in range(N_EVAL):
                r, _ = D.run_episode(env, actor, pah, 'B4', 'det')
                rows.append(dict(run=run, seed=s, episode=k, **r))
        print('evaluated', RUNS[run])
    ev = pd.DataFrame(rows); ev.to_csv(EVAL_CSV, index=False)

def per_seed(df):
    g = df.groupby(['run', 'seed'])
    out = g.apply(lambda x: pd.Series({
        'success': 100 * (x.outcome == 'success').mean(), 'collision': 100 * (x.outcome == 'collision').mean(),
        'timeout': 100 * (x.outcome == 'timeout').mean(),
        't_ave': x.loc[x.outcome == 'success', 'steps'].mean(),
        'obstacle_share': 100 * (x.loc[x.outcome == 'collision', 'collision_kind'] != 'drone-drone').mean(),
        'los_per_episode': x.los_pair_steps.mean()}), include_groups=False).reset_index()
    return out
ps = per_seed(ev)
ps.to_csv(OUT + '/per_seed.csv', index=False)
print(ps.assign(run=ps.run.map(RUNS)).round(2).to_string(index=False))

loaded cached output/eval_episodes.csv
                        run  seed  success  collision  timeout  t_ave  obstacle_share  los_per_episode
Fixed-weight MAPPO (laptop)    42     85.0       15.0      0.0  48.56          100.00             0.12
Fixed-weight MAPPO (laptop)    43     91.0        9.0      0.0  49.20          100.00             0.14
Fixed-weight MAPPO (laptop)    44     87.5       12.5      0.0  49.05           96.00             0.03
Fixed-weight MAPPO (laptop)    45     88.5       11.5      0.0  48.16          100.00             0.28
Fixed-weight MAPPO (laptop)    46     89.0       11.0      0.0  48.99          100.00             0.13
                Plain MAPPO    42     89.0       11.0      0.0  48.81          100.00             0.07
                Plain MAPPO    43     88.5       11.5      0.0  49.29          100.00             0.07
                Plain MAPPO    44     92.0        8.0      0.0  49.51           93.75             0.34
                Plain MAPPO    45 

In [3]:
# Cell 3 -- GATE 3: laptop Fixed-weight MAPPO vs the original Kaggle Fixed-weight MAPPO
kag = pd.read_csv(os.path.join(R.V3, 'results-table', 'output', 'per_seed_stage2.csv'))
kag_fw = kag[kag.method == 'B4'].set_index('seed').success
lap_fw = ps[ps.run == 'fixed-weight-mappo-laptop'].set_index('seed').success
print('Kaggle Fixed-weight MAPPO : {:.1f} ± {:.1f}  per seed {}'.format(kag_fw.mean(), kag_fw.std(ddof=1), kag_fw.round(1).tolist()))
print('Laptop Fixed-weight MAPPO : {:.1f} ± {:.1f}  per seed {}'.format(lap_fw.mean(), lap_fw.std(ddof=1), lap_fw.round(1).tolist()))
gap = abs(lap_fw.mean() - kag_fw.mean())
print('difference of means: {:.2f} points (Kaggle seed-to-seed std {:.2f})'.format(gap, kag_fw.std(ddof=1)))
assert gap <= 2 * kag_fw.std(ddof=1), 'GATE 3 FAILED -- laptop training does not reproduce the Kaggle level; stop and investigate'
print('GATE 3 PASSED')

Kaggle Fixed-weight MAPPO : 89.2 ± 2.0  per seed [87.5, 92.5, 89.5, 87.5, 89.0]
Laptop Fixed-weight MAPPO : 88.2 ± 2.2  per seed [85.0, 91.0, 87.5, 88.5, 89.0]
difference of means: 1.00 points (Kaggle seed-to-seed std 2.05)
GATE 3 PASSED


In [4]:
# Cell 4 -- the comparison (reading rule from PLAN.md Section 6)
def welch(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    diff = a.mean() - b.mean()
    va, vb = a.var(ddof=1) / len(a), b.var(ddof=1) / len(b)
    se = math.sqrt(va + vb)
    df = (va + vb) ** 2 / (va ** 2 / (len(a) - 1) + vb ** 2 / (len(b) - 1))
    t = stats.t.ppf(0.975, df)
    p = stats.ttest_ind(a, b, equal_var=False).pvalue
    return diff, diff - t * se, diff + t * se, p

def verdict(lo, hi):
    if -3 <= lo and hi <= 3 and lo <= 0 <= hi: return 'MATCHES (CI inside ±3 and contains 0)'
    if hi < 0: return 'LOWER (CI entirely below 0)'
    if lo > 0: return 'HIGHER (CI entirely above 0)'
    return 'INCONCLUSIVE'

succ = {r: ps[ps.run == r].sort_values('seed').success.values for r in RUNS}
rows = []
for r in ('plain-mappo', 'plain-mappo-alpha09'):
    d, lo, hi, p = welch(succ[r], succ['fixed-weight-mappo-laptop'])
    rows.append({'Comparison': '{} − Fixed-weight MAPPO (laptop)'.format(RUNS[r]), 'Difference (points)': round(d, 2),
                 '95% CI': '[{:.2f}, {:.2f}]'.format(lo, hi), 'Welch p': round(p, 3), 'Reading': verdict(lo, hi)})
CMP = pd.DataFrame(rows)
print(CMP.to_string(index=False))

summary = []
for r, name in RUNS.items():
    x = ps[ps.run == r]
    summary.append({'Run': name, 'Success %': '{:.1f} ± {:.1f}'.format(x.success.mean(), x.success.std(ddof=1)),
                    'Collision %': '{:.1f} ± {:.1f}'.format(x.collision.mean(), x.collision.std(ddof=1)),
                    'Timeout %': '{:.1f}'.format(x.timeout.mean()), 'T_ave': '{:.1f}'.format(x.t_ave.mean()),
                    '% collisions with obstacles': '{:.0f}'.format(x.obstacle_share.mean()),
                    'Loss-of-separation / episode': '{:.3f}'.format(x.los_per_episode.mean())})
# context: the original Kaggle v3 methods (different hardware -- context only)
for key, name in (('B4', 'Fixed-weight MAPPO (Kaggle, v3)'), ('H', 'Hand-coded τ-rule MAPPO (Kaggle, v3)'), ('M', 'MAPPO + PAH (Kaggle, v3)')):
    x = kag[kag.method == key]
    summary.append({'Run': name, 'Success %': '{:.1f} ± {:.1f}'.format(x.success.mean(), x.success.std(ddof=1)),
                    'Collision %': '{:.1f} ± {:.1f}'.format(x.collision.mean(), x.collision.std(ddof=1)),
                    'Timeout %': '{:.1f}'.format(x.timeout.mean()), 'T_ave': '{:.1f}'.format(x.t_ave.mean()),
                    '% collisions with obstacles': '—', 'Loss-of-separation / episode': '—'})
SUM = pd.DataFrame(summary)
print(); print(SUM.to_string(index=False))

wall = [json.load(open(os.path.join('runs', r, 'output-seed{}'.format(s), 'wall_time.json'))) for r in RUNS for s in SEEDS]
wm = [w['wall_minutes'] for w in wall]
print('\nwall time per full 8000-episode run (8 runs in parallel): {:.1f}–{:.1f} min, median {:.1f}'.format(min(wm), max(wm), np.median(wm)))

with open(OUT + '/results.md', 'w') as f:
    f.write('## Success, Stage 2 (deterministic, 200 episodes, mean ± std over 5 seeds)\n\n' + SUM.to_markdown(index=False) + '\n\n')
    f.write('## Comparison (reading rule from PLAN.md)\n\n' + CMP.to_markdown(index=False) + '\n\n')
    f.write('Wall time per full run (8 in parallel on the laptop): {:.1f}–{:.1f} min, median {:.1f}.\n'.format(min(wm), max(wm), np.median(wm)))
print('saved', OUT + '/results.md')

                                               Comparison  Difference (points)        95% CI  Welch p      Reading
                Plain MAPPO − Fixed-weight MAPPO (laptop)                  1.7 [-1.07, 4.47]    0.188 INCONCLUSIVE
Plain MAPPO, α = 0.9 reward − Fixed-weight MAPPO (laptop)                  1.2 [-1.82, 4.22]    0.385 INCONCLUSIVE

                                 Run  Success % Collision % Timeout % T_ave % collisions with obstacles Loss-of-separation / episode
         Fixed-weight MAPPO (laptop) 88.2 ± 2.2  11.8 ± 2.2       0.0  48.8                          99                        0.140
                         Plain MAPPO 89.9 ± 1.4  10.1 ± 1.4       0.0  49.2                          96                        0.157
         Plain MAPPO, α = 0.9 reward 89.4 ± 1.9  10.6 ± 1.9       0.0  48.7                          94                        0.141
     Fixed-weight MAPPO (Kaggle, v3) 89.2 ± 2.0  10.8 ± 2.0       0.0  49.1                           —                   

In [5]:
# Cell 5 -- figures (white background, full names)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white', 'savefig.facecolor': 'white',
                     'font.family': 'DejaVu Sans', 'font.size': 9.5, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'axes.axisbelow': True, 'grid.alpha': 0.25})
COL = {'fixed-weight-mappo-laptop': '#808080', 'plain-mappo': '#2a78d6', 'plain-mappo-alpha09': '#1baf7a'}
WRAP = {'fixed-weight-mappo-laptop': 'Fixed-weight MAPPO\n(laptop)', 'plain-mappo': 'Plain MAPPO',
        'plain-mappo-alpha09': 'Plain MAPPO,\nα = 0.9 reward'}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw=dict(width_ratios=[1, 1.4]))
ax = axes[0]
for i, r in enumerate(RUNS):
    v = succ[r]
    ax.bar(i, v.mean(), 0.6, yerr=v.std(ddof=1), color=COL[r], edgecolor='white', linewidth=2, capsize=4, error_kw=dict(lw=1))
    ax.scatter(np.full(len(v), i) + np.linspace(-0.12, 0.12, len(v)), v, s=14, color='#222', zorder=3)
    ax.text(i, v.mean() + v.std(ddof=1) + 1.0, '{:.1f}%'.format(v.mean()), ha='center', va='bottom', fontsize=9)
kf = kag_fw.values
ax.axhspan(kf.mean() - kf.std(ddof=1), kf.mean() + kf.std(ddof=1), color='#808080', alpha=0.12, lw=0)
ax.axhline(kf.mean(), color='#808080', lw=1.2, ls='--')
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([0], [0], color='#808080', lw=1.2, ls='--', label='Kaggle Fixed-weight MAPPO (v3): {:.1f}% (band = ± std)'.format(kf.mean()))],
          loc='upper right', frameon=False, fontsize=8)
ax.set_xticks(range(len(RUNS))); ax.set_xticklabels([WRAP[r] for r in RUNS], fontsize=8.6)
ax.set_ylim(70, 100); ax.set_ylabel('Mission success rate (%)')
ax.set_title('Final models (deterministic, 200 episodes;\nbars = mean ± std over 5 seeds, dots = seeds)', fontsize=10)

ax = axes[1]
GRID = np.arange(100, 7901, 100)
for r in RUNS:
    ys = []
    for s in SEEDS:
        h = json.load(open(os.path.join('runs', r, 'output-seed{}'.format(s), 'history.json')))
        y = np.interp(GRID, h['episode'], [100 * v for v in h['success']])
        ys.append(np.array([y[max(0, i - 4): i + 1].mean() for i in range(len(y))]))
    ys = np.array(ys); m, sd = ys.mean(0), ys.std(0, ddof=1)
    ax.fill_between(GRID, m - sd, m + sd, color=COL[r], alpha=0.15, lw=0)
    ax.plot(GRID, m, color=COL[r], lw=2, label=RUNS[r])
ax.set_xlim(0, 8000); ax.set_ylim(70, 100); ax.set_xlabel('Training episode'); ax.set_ylabel('Success rate (%)')
ax.set_title('Training curves (training-time evaluation: stochastic, 20 episodes;\n5-checkpoint moving average, mean ± std over 5 seeds)', fontsize=10)
ax.legend(frameon=False, fontsize=8.5, loc='lower right')
fig.suptitle('Phase 1a — does Plain MAPPO (no mission/safety mixing) match Fixed-weight MAPPO? Stage 2 (5 drones, 5 obstacles)', fontsize=10.5)
plt.tight_layout(rect=(0, 0, 1, 0.93))
fig.savefig(OUT + '/plain_mappo_check.png', dpi=200, bbox_inches='tight'); fig.savefig(OUT + '/plain_mappo_check.svg', bbox_inches='tight')
plt.close(fig); print('saved', OUT + '/plain_mappo_check.png')

saved output/plain_mappo_check.png
